In [10]:
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


In [11]:
random.seed(42)
epochs = 30
batch_size = 32
learning_rate = 0.01

train_transform = transforms.Compose(
    [
        transforms.Resize((28, 28)),
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomRotation(10), 
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]
)

test_transform = transforms.Compose(
    [
        transforms.Resize((28, 28)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]
)


In [12]:
train_dataset = ImageFolder(root=r'auto data/train', transform=train_transform)
test_dataset = ImageFolder(root=r'auto data/test', transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset , batch_size = batch_size , shuffle = False)

In [13]:
# تقسيم بيانات التدريب إلى 80% للتدريب و 20% للتقييم (Validation)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_set, val_set = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [14]:
print("Classes:", train_dataset.classes)
print(f"Training images: {len(train_set)}")
print(f"Validation images: {len(val_set)}")
print(f"Test images: {len(test_dataset)}")

Classes: ['F', 'L', 'O', 'W']
Training images: 13492
Validation images: 3374
Test images: 908


In [15]:
images, labels = next(iter(train_loader))
print(f"Batch images shape: {images.shape}")
print(f"Batch labels shape: {labels.shape}")

Batch images shape: torch.Size([32, 1, 28, 28])
Batch labels shape: torch.Size([32])


In [16]:
class auto_movement(nn.Module):
    def __init__(self):
        super(auto_movement, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(16)
        self.maxpool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(32)
        
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 4)   
        
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.maxpool(x)
        x = x.view(-1, 32 * 7 * 7) 
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [17]:
model = auto_movement()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

In [18]:
total_steps = len(train_loader)

for epoch in range(epochs):
   
    model.train() 
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for i, (images, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    scheduler.step() 
    
    model.eval() 
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = loss_function(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    
    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train
    
    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct_val / total_val
    
    
    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train - Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"Val   - Loss: {avg_val_loss:.4f} | Acc: {val_acc:.2f}%")
    print("-" * 40)

Epoch [1/30]
Train - Loss: 1.1228 | Acc: 48.32%
Val   - Loss: 0.6984 | Acc: 74.60%
----------------------------------------
Epoch [2/30]
Train - Loss: 0.6674 | Acc: 73.01%
Val   - Loss: 0.3850 | Acc: 85.57%
----------------------------------------
Epoch [3/30]
Train - Loss: 0.4443 | Acc: 82.19%
Val   - Loss: 0.2755 | Acc: 88.97%
----------------------------------------
Epoch [4/30]
Train - Loss: 0.3092 | Acc: 88.45%
Val   - Loss: 0.1598 | Acc: 94.61%
----------------------------------------
Epoch [5/30]
Train - Loss: 0.2309 | Acc: 91.65%
Val   - Loss: 0.1231 | Acc: 95.58%
----------------------------------------
Epoch [6/30]
Train - Loss: 0.1944 | Acc: 93.37%
Val   - Loss: 0.0793 | Acc: 97.24%
----------------------------------------
Epoch [7/30]
Train - Loss: 0.1607 | Acc: 94.41%
Val   - Loss: 0.0675 | Acc: 97.57%
----------------------------------------
Epoch [8/30]
Train - Loss: 0.1437 | Acc: 95.00%
Val   - Loss: 0.0540 | Acc: 98.37%
----------------------------------------
Epoch [9

In [19]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        outputs = model(images)
        _,predicted= torch.max(outputs.data,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        acc= 100 * (correct/total)

In [20]:
print(f'Accuracy of the model on the test images: {acc:.2f}%')

Accuracy of the model on the test images: 100.00%


In [21]:
torch.save(model.state_dict(), 'hand_model_4classes.pth')
print("Model saved successfully!")

Model saved successfully!
